# Module 09 — Data Stream Restore Runbook

Module 08 catalogued the gotchas. This module provides the solution: three helper
functions that wrap the raw ES API and handle every gotcha automatically.

| Helper | What it does |
|--------|--------------|
| `precheck_data_stream_restore` | Read-only pre-flight: checks snapshot, templates, conflicts. Never modifies anything. |
| `safe_snapshot_data_stream` | Takes a complete, self-contained snapshot (always `include_global_state=True`). |
| `safe_restore_data_stream` | Runs pre-flight, deletes existing streams, restores, forces post-restore rollover. |

## Scenario

We will simulate a realistic disaster-recovery workflow:

| Step | Section |
|------|---------|
| Create a data stream and ingest data | 1 |
| Take a safe snapshot | 2 |
| Run pre-flight checks on the snapshot | 3 |
| Simulate data loss (delete the stream) | 4 |
| Restore with the helper and verify | 5 |
| Demonstrate the helper catching problems | 6 |
| Side-by-side restore using rename | 7 |
| Cleanup | 8 |

In [1]:
from helpers import *

es = get_client()
wait_for_green(es)
register_fs_repo(es)

✓ Cluster health: yellow

✓ Repository 'training-fs-repo' registered at /usr/share/elasticsearch/snapshots

---
## 1. Create a Data Stream and Ingest Data

In [2]:
heading('1. Setup — templates and data stream')

# Cleanup from any previous run
for ds in ['runbook-logs-app', 'runbook-logs-sidebyside']:
    try:
        es.indices.delete_data_stream(name=ds)
    except Exception:
        pass
for tpl in ['runbook-logs-template', 'runbook-logs-settings']:
    try:
        es.indices.delete_index_template(name=tpl)
    except Exception:
        pass
    try:
        es.cluster.delete_component_template(name=tpl)
    except Exception:
        pass

# Component template
es.cluster.put_component_template(
    name='runbook-logs-settings',
    body={
        'template': {
            'settings': {'number_of_shards': 1, 'number_of_replicas': 0},
            'mappings': {
                'properties': {
                    '@timestamp': {'type': 'date'},
                    'message':    {'type': 'text'},
                    'level':      {'type': 'keyword'},
                    'service':    {'type': 'keyword'},
                }
            },
        }
    },
)

# Composable index template
es.indices.put_index_template(
    name='runbook-logs-template',
    body={
        'index_patterns': ['runbook-logs-*'],
        'data_stream': {},
        'composed_of': ['runbook-logs-settings'],
        'priority': 200,
    },
)
success('Templates created')

# Ingest 50 documents across two simulated services
import datetime
base = datetime.datetime(2025, 6, 1, 0, 0, 0)
for i in range(50):
    es.index(
        index='runbook-logs-app',
        document={
            '@timestamp': (base + datetime.timedelta(minutes=i * 5)).isoformat(),
            'message': f'Request processed ({i})',
            'level': 'INFO' if i % 10 != 0 else 'ERROR',
            'service': 'payments' if i % 2 == 0 else 'auth',
        },
    )
es.indices.refresh(index='runbook-logs-app')

count = es.count(index='runbook-logs-app')['count']
ds = es.indices.get_data_stream(name='runbook-logs-app')['data_streams'][0]
success(f'Data stream runbook-logs-app: {count} documents')
info(f'  Write index: {ds["indices"][-1]["index_name"]}')
info(f'  Generation : {ds["generation"]}')

────────────────────────────────────── 1. Setup — templates and data stream ───────────────────────────────────────

✓ Templates created

✓ Data stream runbook-logs-app: 50 documents

ℹ   Write index: .ds-runbook-logs-app-2026.04.12-000001

ℹ   Generation : 1

---
## 2. Take a Safe Snapshot

`safe_snapshot_data_stream` always uses `include_global_state=True` so the
composable template travels with the snapshot.  It also warns if any of the
streams look Fleet-managed.

In [3]:
delete_snapshot_if_exists(es, FS_REPO_NAME, 'runbook-snap')

snap = safe_snapshot_data_stream(
    client=es,
    repository=FS_REPO_NAME,
    snapshot_name='runbook-snap',
    streams=['runbook-logs-app'],
)

ℹ Deleted existing snapshot 'runbook-snap'

───────────────────────────────── Snapshotting data streams: ['runbook-logs-app'] ─────────────────────────────────

✓ Snapshot 'runbook-snap' — state: SUCCESS

ℹ   Data streams   : ['runbook-logs-app']

ℹ   Backing indices: 23 index(es)

ℹ   Global state   : included (templates travel with snapshot)

---
## 3. Run Pre-flight Checks

`precheck_data_stream_restore` is read-only — it never modifies anything.
Run it before every restore to catch problems early.

In [4]:
heading('3. Pre-flight checks — all clear expected')

precheck_data_stream_restore(
    client=es,
    repository=FS_REPO_NAME,
    snapshot='runbook-snap',
    streams=['runbook-logs-app'],
)

──────────────────────────────────── 3. Pre-flight checks — all clear expected ────────────────────────────────────

──────────────────────────────────────────────── Pre-flight checks ────────────────────────────────────────────────

✓ Snapshot 'runbook-snap' found — state: SUCCESS

Stream: runbook-logs-app

✓   'runbook-logs-app' is listed as a data stream in the snapshot

✓   Composable template found for 'runbook-logs-app'

⚠   Data stream 'runbook-logs-app' already exists on this cluster. It must be deleted before restoring under the 
same name.

Pre-flight PASSED WITH WARNINGS — 1 warning(s). Review before proceeding.

True

---
## 4. Simulate Data Loss

In [5]:
heading('4. Simulate data loss — delete the data stream')

pre_count = es.count(index='runbook-logs-app')['count']
es.indices.delete_data_stream(name='runbook-logs-app')
warn(f'Data stream deleted ({pre_count} documents lost)')

# Verify it is gone
try:
    es.indices.get_data_stream(name='runbook-logs-app')
    warn('Stream still exists (unexpected)')
except Exception:
    success('Confirmed: runbook-logs-app does not exist')

───────────────────────────────── 4. Simulate data loss — delete the data stream ──────────────────────────────────

⚠ Data stream deleted (50 documents lost)

✓ Confirmed: runbook-logs-app does not exist

---
## 5. Restore with the Helper and Verify

`safe_restore_data_stream` handles the full workflow:
1. Runs pre-flight checks
2. Verifies the composable template exists (hard stop if not)
3. Deletes the existing stream if present
4. Restores using the data stream name (never raw `.ds-*` indices)
5. Forces a rollover to establish a clean write boundary

In [6]:
results = safe_restore_data_stream(
    client=es,
    repository=FS_REPO_NAME,
    snapshot='runbook-snap',
    streams=['runbook-logs-app'],
    delete_existing=True,
    post_rollover=True,
)

# Verify the data is back and the stream is healthy
es.indices.refresh(index='runbook-logs-app')
post_count = es.count(index='runbook-logs-app')['count']
ds = es.indices.get_data_stream(name='runbook-logs-app')['data_streams'][0]

console.print(f'\n[bold]Post-restore verification:[/bold]')
info(f'  Documents recovered : {post_count}')
info(f'  Backing indices     : {[i["index_name"] for i in ds["indices"]]}')
info(f'  Write index         : {ds["indices"][-1]["index_name"]}  ← new, post-restore')
info(f'  Generation          : {ds["generation"]}')

# Confirm new documents can be indexed
es.index(
    index='runbook-logs-app',
    document={'@timestamp': '2025-06-30T12:00:00', 'message': 'Post-restore write test', 'level': 'INFO', 'service': 'payments'},
)
success('Post-restore write succeeded — stream is healthy')

──────────────────────────────────────────────── Pre-flight checks ────────────────────────────────────────────────

✓ Snapshot 'runbook-snap' found — state: SUCCESS

Stream: runbook-logs-app

✓   'runbook-logs-app' is listed as a data stream in the snapshot

✓   Composable template found for 'runbook-logs-app'

...

ℹ   Documents recovered : 50

ℹ   Backing indices     : ['.ds-runbook-logs-app-2026.04.12-000001', '.ds-runbook-logs-app-2026.04.12-000002']

ℹ   Write index         : .ds-runbook-logs-app-2026.04.12-000002  ← new, post-restore

ℹ   Generation          : 2

✓ Post-restore write succeeded — stream is healthy

---
## 6. Demonstrate the Helper Catching Problems

Here we show what happens when the helper is asked to do something unsafe.
Each case would silently fail or corrupt state with the raw API.

### 6a — Missing template blocks restore

In [7]:
heading('6a. Helper blocks restore when template is missing')

# ES does not allow deleting a template that is in use by a data stream.
# Delete the stream first, then the template — simulating a cross-cluster
# restore where neither exists on the target.
es.indices.delete_data_stream(name='runbook-logs-app')
es.indices.delete_index_template(name='runbook-logs-template')
warn('Deleted data stream and composable template to simulate bare-target restore')

try:
    safe_restore_data_stream(
        client=es,
        repository=FS_REPO_NAME,
        snapshot='runbook-snap',
        streams=['runbook-logs-app'],
    )
except RuntimeError as e:
    success('Helper blocked the restore as expected:')
    console.print(f'  [yellow]{e}[/yellow]')

# Fix: create the template, then the helper will succeed
es.indices.put_index_template(
    name='runbook-logs-template',
    body={
        'index_patterns': ['runbook-logs-*'],
        'data_stream': {},
        'composed_of': ['runbook-logs-settings'],
        'priority': 200,
    },
)
success('Template re-created — restore will now succeed')

safe_restore_data_stream(
    client=es,
    repository=FS_REPO_NAME,
    snapshot='runbook-snap',
    streams=['runbook-logs-app'],
)


─────────────────────────────── 6a. Helper blocks restore when template is missing ────────────────────────────────

⚠ Deleted data stream and composable template to simulate bare-target restore

──────────────────────────────────────────────── Pre-flight checks ────────────────────────────────────────────────

✓ Snapshot 'runbook-snap' found — state: SUCCESS

Stream: runbook-logs-app

...

ℹ   Backing    : ['.ds-runbook-logs-app-2026.04.12-000001']

✓   Rollover complete — new write index: .ds-runbook-logs-app-2026.04.12-000002

──────────────────────────────────────────────── Restore complete ─────────────────────────────────────────────────

✓   runbook-logs-app → runbook-logs-app  |  50 docs  |  write index: .ds-runbook-logs-app-2026.04.12-000002

{'runbook-logs-app': {'target': 'runbook-logs-app',
  'doc_count': 50,
  'generation': 1,
  'write_index': '.ds-runbook-logs-app-2026.04.12-000002'}}

### 6b — Pre-flight warns when stream already exists

In [8]:
heading('6b. Pre-flight warns about existing stream')

# runbook-logs-app still exists from section 5 — pre-flight should warn
precheck_data_stream_restore(
    client=es,
    repository=FS_REPO_NAME,
    snapshot='runbook-snap',
    streams=['runbook-logs-app'],
)
info('The helper will auto-delete it when delete_existing=True (the default)')

─────────────────────────────────── 6b. Pre-flight warns about existing stream ────────────────────────────────────

──────────────────────────────────────────────── Pre-flight checks ────────────────────────────────────────────────

✓ Snapshot 'runbook-snap' found — state: SUCCESS

Stream: runbook-logs-app

✓   'runbook-logs-app' is listed as a data stream in the snapshot

✓   Composable template found for 'runbook-logs-app'

⚠   Data stream 'runbook-logs-app' already exists on this cluster. It must be deleted before restoring under the 
same name.

Pre-flight PASSED WITH WARNINGS — 1 warning(s). Review before proceeding.

ℹ The helper will auto-delete it when delete_existing=True (the default)

### 6c — Pre-flight warns about Fleet-managed streams

In [9]:
heading('6c. Pre-flight flags a Fleet-managed stream name')

# We do not actually restore — just show what the pre-flight reports
# Use a made-up fleet-style name so we don't touch real fleet indices
precheck_data_stream_restore(
    client=es,
    repository=FS_REPO_NAME,
    snapshot='runbook-snap',
    streams=['logs-myapp-default'],   # fleet-style name
)

──────────────────────────────── 6c. Pre-flight flags a Fleet-managed stream name ─────────────────────────────────

──────────────────────────────────────────────── Pre-flight checks ────────────────────────────────────────────────

✓ Snapshot 'runbook-snap' found — state: SUCCESS

Stream: logs-myapp-default

✗ 'logs-myapp-default' not found in snapshot at all

✓   Composable template found for 'logs-myapp-default'

✓   'logs-myapp-default' does not exist — safe to restore

⚠   'logs-myapp-default' looks like a Fleet-managed stream. Do NOT restore with include_global_state=True — 
reinstall Fleet integrations on the target first.

Pre-flight FAILED — 1 blocker(s), 1 warning(s). Fix blockers before restoring.

False

---
## 7. Side-by-Side Restore Using Rename

Pass `rename_to` to restore the snapshot under a different name, keeping the live
stream running.  The pre-flight check verifies the new name has a matching template
before any destructive action is taken.

In [10]:
heading('7. Side-by-side restore — keep live stream, restore a point-in-time copy')

try:
    es.indices.delete_data_stream(name='runbook-logs-sidebyside')
except Exception:
    pass

results = safe_restore_data_stream(
    client=es,
    repository=FS_REPO_NAME,
    snapshot='runbook-snap',
    streams=['runbook-logs-app'],
    rename_to={'runbook-logs-app': 'runbook-logs-sidebyside'},
    delete_existing=True,
    post_rollover=False,   # side-by-side copy — no need for a write boundary
)

es.indices.refresh(index='runbook-logs-sidebyside')
live_count = es.count(index='runbook-logs-app')['count']
copy_count = es.count(index='runbook-logs-sidebyside')['count']

info(f'runbook-logs-app       : {live_count} docs (live stream, still receiving writes)')
info(f'runbook-logs-sidebyside: {copy_count} docs (point-in-time copy from snapshot)')

──────────────────── 7. Side-by-side restore — keep live stream, restore a point-in-time copy ─────────────────────

──────────────────────────────────────────────── Pre-flight checks ────────────────────────────────────────────────

✓ Snapshot 'runbook-snap' found — state: SUCCESS

Stream: runbook-logs-app  →  runbook-logs-sidebyside

✓   'runbook-logs-app' is listed as a data stream in the snapshot

...

ℹ   Backing    : ['.ds-runbook-logs-sidebyside-2026.04.12-000001']

──────────────────────────────────────────────── Restore complete ─────────────────────────────────────────────────

✓   runbook-logs-app → runbook-logs-sidebyside  |  50 docs  |  write index: 
.ds-runbook-logs-sidebyside-2026.04.12-000001

ℹ runbook-logs-app       : 50 docs (live stream, still receiving writes)

ℹ runbook-logs-sidebyside: 50 docs (point-in-time copy from snapshot)

---
## 8. Cleanup

In [11]:
heading('8. Cleanup')

for ds in ['runbook-logs-app', 'runbook-logs-sidebyside']:
    try:
        es.indices.delete_data_stream(name=ds)
        success(f'Deleted data stream: {ds}')
    except Exception:
        pass

for tpl in ['runbook-logs-template', 'runbook-logs-settings']:
    try:
        es.indices.delete_index_template(name=tpl)
        success(f'Deleted index template: {tpl}')
    except Exception:
        pass
    try:
        es.cluster.delete_component_template(name=tpl)
        success(f'Deleted component template: {tpl}')
    except Exception:
        pass

delete_snapshot_if_exists(es, FS_REPO_NAME, 'runbook-snap')
success('Cleanup complete')

─────────────────────────────────────────────────── 8. Cleanup ────────────────────────────────────────────────────

✓ Deleted data stream: runbook-logs-app

✓ Deleted data stream: runbook-logs-sidebyside

✓ Deleted index template: runbook-logs-template

✓ Deleted component template: runbook-logs-settings

ℹ Deleted existing snapshot 'runbook-snap'

✓ Cleanup complete

---
## What the Helpers Handle Automatically

| Module 08 Gotcha | Handled by |
|------------------|-----------|
| Template missing → rollover breaks | `safe_restore` hard-stops before restore |
| Data stream must be deleted first | `safe_restore` deletes it automatically (`delete_existing=True`) |
| `include_global_state=False` loses templates | `safe_snapshot` always uses `True` |
| Renaming requires matching template | Pre-flight checks the *target* name, not just the source |
| Restoring `.ds-*` directly loses stream | `safe_restore` always uses the stream name |
| No clean write boundary after restore | `safe_restore` forces rollover (`post_rollover=True`) |
| Fleet-managed streams need special care | Pre-flight warns and explains the risk |

## What Still Requires Manual Action

| Gap | Why |
|-----|-----|
| Restoring templates independently from global state | ES has no "restore just templates" API |
| Dry-run restore | ES has no preview/simulation mode for restore |
| Fleet streams: integrations must be reinstalled first | Fleet owns those templates; helpers can only warn |
| Cross-cluster restore to a bare cluster | Target must have the template pre-created manually |